# CourtVision on Colab

1. **Runtime → Change runtime type → T4 GPU.**
2. In the **Secrets** panel (key icon), add `ROBOFLOW_API_KEY` and `COURTVISION_PASSWORD`, and turn on notebook access for both.
3. Put the roster CSV on Drive at `MyDrive/CS496/court vision/rosters.csv`. Videos are found anywhere under `MyDrive/CS496/court vision/` (shortcuts work).

Run the cells from the top. If the install cell says to restart the session, do it and run the cells again.

## 1. Drive and GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

# Your project folder on Drive (case-sensitive). Change this if your folder is elsewhere.
DRIVE = '/content/drive/MyDrive/CS496/court vision'

import os
if not os.path.isdir(DRIVE):
    root, found = '/content/drive/MyDrive', []
    for path, dirs, _ in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        if path.count(os.sep) - root.count(os.sep) >= 3:
            dirs[:] = []
        found += [os.path.join(path, d) for d in dirs
                  if 'courtvision' in d.lower().replace(' ', '').replace('_', '')]
    hint = ('Folders that look right:\n  ' + '\n  '.join(found)) if found else (
        'No folder named like "courtvision" in My Drive. Top-level folders:\n  '
        + '\n  '.join(sorted(os.listdir(root))))
    raise FileNotFoundError(f'DRIVE folder not found: {DRIVE}\n{hint}\n'
                            'Set DRIVE above to the right path and run this cell again.')
print('Using', DRIVE)

## 2. Install CourtVision

In [ ]:
import os
from google.colab import userdata

BRANCH = 'dev-jason'  # branch with the latest work
REPO = 'https://github.com/Mythventor/CourtVision.git'

if not os.path.exists('/content/CourtVision'):
    !git clone -q -b {BRANCH} {REPO} /content/CourtVision
else:
    !git -C /content/CourtVision fetch -q && git -C /content/CourtVision checkout -q {BRANCH} && git -C /content/CourtVision pull -q
%cd /content/CourtVision
!python --version
# uv resolves the large inference/torch dependency set more reliably (and faster) than pip
!pip install -q uv
import sys
# Install into this notebook's own Python (not whichever python is first on PATH)
!uv pip install --python {sys.executable} -q -e ".[cv,app]"

# An editable install is only picked up when Python restarts, so add the source folder
# to the path for this session too.
SRC = '/content/CourtVision/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import courtvision
print('courtvision', courtvision.__version__, 'installed for', sys.executable)

for name in ('ROBOFLOW_API_KEY', 'COURTVISION_PASSWORD'):
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        print(f'Secret {name} not set')

## 3. Config on Drive

Created once at `MyDrive/CS496/court vision/config.yaml`. It only needs the settings that differ from `configs/default.yaml`; edit it on Drive (e.g. to add `court.template`).

In [ ]:
import yaml

CONFIG = f'{DRIVE}/config.yaml'
if not os.path.exists(CONFIG):
    with open(CONFIG, 'w') as f:  # yaml.safe_dump quotes paths with spaces correctly
        yaml.safe_dump({'video_root': DRIVE,
                        'output_dir': f'{DRIVE}/outputs',
                        'roster_csv': f'{DRIVE}/rosters.csv'}, f, sort_keys=False)
print(open(CONFIG).read())

from courtvision.config import load_config
config = load_config(CONFIG)

## 4. Find the videos

Searches everything under `video_root` (your `DRIVE` folder), including shortcuts, and groups videos by day folder. Set `PICK` to the number of the video to use in sections 5 and 6. The first run can take a minute while Drive lists the folders.

In [ ]:
from courtvision.library import describe_folder, list_videos

videos = list_videos(config['video_root'], exclude=config['output_dir'])
all_videos = []
for folder, files in videos.items():
    print(folder)
    for p in files:
        print(f'  [{len(all_videos)}] {p.name}  ({p.stat().st_size / 1e9:.1f} GB)')
        all_videos.append(p)

PICK = 0  # number from the list above
VIDEO = str(all_videos[PICK]) if all_videos else None
if VIDEO:
    print('\nVIDEO =', VIDEO)
    !ffprobe -v error -show_entries format=duration:format_tags=creation_time:stream=codec_name,width,height,r_frame_rate -of default=nw=1 "{VIDEO}"
else:
    print('No videos found under', config['video_root'], '- folders there:')
    print('\n'.join(describe_folder(config['video_root'], depth=3)))

## 5. Check the detection model on one frame

Compare the class names it prints with `detection.classes` in the config.

In [ ]:
import matplotlib.pyplot as plt
from courtvision.preview import preview_detections

T = 120  # seconds into the video
img, classes = preview_detections(VIDEO, T, config)
print('Classes in this frame:', classes)
plt.figure(figsize=(16, 9)); plt.imshow(img); plt.axis('off'); plt.show()

## 6. Court keypoints (one-time setup for the 2D panel)

This numbers the court keypoints the model finds (yellow = confident, red = not). Send the image and the printed list to Claude to fill in `court.template`, or fill it in yourself: one `[x_ft, y_ft]` per keypoint index, `null` for ones to skip.

In [ ]:
from courtvision.preview import preview_keypoints

img, points = preview_keypoints(VIDEO, T, config)
for i, x, y, c in points:
    print(f'{i:2d}: x={x:7.1f} y={y:7.1f} conf={c:.2f}')
plt.figure(figsize=(16, 9)); plt.imshow(img); plt.axis('off'); plt.show()

## 7. Start the app

Open the `https://….gradio.live` link it prints and log in as `courtvision` with your password. Processed videos are saved to `MyDrive/CS496/court vision/outputs`, so a quarter only has to be processed once. Try a 2-minute range first to check speed.

In [ ]:
# Same Python as this notebook, with the source folder on the path
!PYTHONPATH="{SRC}" {sys.executable} -m courtvision.app --share --config "{CONFIG}"